# mjorbit P4.3 v2: bar-pose estimator, calibrated sensors

v2 data: 500 episodes, images every 0.25 s (wrist 128, bus 192, lidar 128), sensor mounts calibrated to about 1 mm and 0.1 deg. Training uses photometric augmentation.

Before running: **Runtime > Change runtime type > T4 GPU**. Drive layout: `MyDrive/mjorbit_p4/data_v2/` (10 shards + SHA256SUMS) and `MyDrive/mjorbit_p4/code_v2/` (scripts). Results go to `MyDrive/mjorbit_p4/pose_v2/`.

If the session disconnects, run all cells again: training resumes from the last finished epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/mjorbit_p4'
!ls -la {D} {D}/data_v2 {D}/code_v2

In [ ]:
# copy the shards to the local disk, check them, unpack (skipped if already packed in this session)
import os
if not os.path.exists('/content/demos_v2_pack/index.json'):
    !mkdir -p /content/shards /content/demos_v2
    !cp {D}/data_v2/* /content/shards/
    !cd /content/shards && sha256sum -c SHA256SUMS
    !for f in /content/shards/demos_v2_*.tar; do tar -xf $f -C /content/demos_v2 && rm $f; done
    !ls /content/demos_v2 | wc -l

In [ ]:
# pack into flat arrays (about 17 GB), one frame per image
!mkdir -p /content/code && cp {D}/code_v2/*.py /content/code/
if not os.path.exists('/content/demos_v2_pack/index.json'):
    !cd /content/code && python pack_data.py /content/demos_v2 /content/demos_v2_pack 6
    !rm -rf /content/demos_v2
!ls -la /content/demos_v2_pack

In [ ]:
# train 30 epochs with augmentation; checkpoint and log go to Drive after every epoch
!mkdir -p {D}/pose_v2
!cd /content/code && python train_pose.py --pack /content/demos_v2_pack --out {D}/pose_v2 --epochs 30 --workers 4 --augment 1 2>&1 | tee -a {D}/pose_v2/train_colab.log

In [ ]:
# per-epoch validation error: median mm overall, settle, reach, approach; minutes
import json
for r in json.load(open(f'{D}/pose_v2/log.json')):
    print(r['epoch'], *(round(r[k]['pos_mm_med'], 1) for k in ('all', 'settle', 'reach', 'approach')), r['min'])